# Example-12: Joined frequency (direct computation)

In [1]:
# Import

import numpy
import torch
import nufft
import yaml

import sys
sys.path.append('..')

from harmonica.util import LENGTH
from harmonica.statistics import weighted_mean, weighted_variance
from harmonica.window import Window
from harmonica.data import Data
from harmonica.frequency import Frequency
from harmonica.filter import Filter

import matplotlib.pyplot as plt

torch.set_printoptions(precision=12, sci_mode=True)
print(torch.cuda.is_available())
print(torch.get_num_threads())

False
8


In [2]:
# Set data type and device

dtype = torch.float64
device = torch.device('cpu')

In [3]:
# Set window

w = Window(4096, name='cosine_window', order=1.0, dtype=dtype, device=device)

# Load test TbT data from file (linear lattice without noise)

d = Data.from_file(54, w, '../virtual_tbt.npy')

# Remove mean and apply window

d.window_remove_mean()
d.window_apply()

# Compute reference frequency

f = Frequency(d)
f('parabola')
ref = f.frequency.mean()
print(ref.item())

# Generate mixed signal

keep = 32
data = d.data[:, :keep]
data = (data - data.mean(1).reshape(-1, 1))/data.std(1).reshape(-1, 1)
data = d.make_signal(keep, data)

# Generate window and TbT for mixed signal

w = Window(len(data), name='cosine_window', order=1.0, dtype=dtype, device=device)
d = Data.from_data(w, data.reshape(1, -1))
d.window_apply()

# Compute frequency

f = Frequency(d)
f('parabola', f_range=(8.5/54, 8.6/54))
res = 9.0 - 54*f.frequency.mean()
print(res.item())

# Compare

print((ref - res).item())

# Note, the above is also avalible as a separate method (work container is used)

w = Window(4096, name='cosine_window', order=1.0, dtype=dtype, device=device)
d = Data.from_file(54, w, '../virtual_tbt.npy')
f = Frequency(d)
*_, res = 9.0 - f.compute_joined_frequency(length=keep, f_range=(8.5, 8.6), order=1.0)
print(res.item())

# Clean

del w
del data
del d
del f
if device != torch.device('cpu'):
    torch.cuda.synchronize()
    torch.cuda.empty_cache()

0.46311690126299115
0.4631161977901481
7.034728430332926e-07
0.4631161977901481
